# 3D Mesh Modeling and Manipulation

In [1]:
# Install once per runtime (safe to rerun)
!pip install -q manifold3d mapbox_earcut numpy-stl shapely solidpython trimesh

## 0. Setup: Imports and Utility Functions

This section loads required dependencies and defines helper functions for mesh loading, rotation, plotting, and GIF export.

In [6]:
import io
import imageio
import numpy as np
import trimesh

from ipywidgets import (
  interactive,
  IntSlider,
  Output,
  VBox
)
from IPython.display import (
  clear_output,
  display,
  Image
)
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path
from scipy.spatial import ConvexHull
import requests
import zipfile
import shutil


OUTPUT_DIR = Path("./sample_meshes")

if not OUTPUT_DIR.exists() or not any(OUTPUT_DIR.iterdir()):
  repo_zip_url = "https://github.com/emanuelazcona/3d_mod_plus_data_viz/archive/refs/heads/main.zip"
  resp = requests.get(repo_zip_url)
  resp.raise_for_status()
  with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    prefix = "3d_mod_plus_data_viz-main/sample_meshes/"
    members = [m for m in z.namelist() if m.startswith(prefix)]
    if not members:
      raise FileNotFoundError("sample_meshes not found in repository archive")
    for member in members:
      rel_path = Path(member).relative_to(prefix)
      target = OUTPUT_DIR / rel_path
      if member.endswith("/"):
        target.mkdir(parents=True, exist_ok=True)
      else:
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(member) as src, open(target, "wb") as dst:
          shutil.copyfileobj(src, dst)

def resolve_output_path(filename: str) -> str:
  OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
  return str((OUTPUT_DIR / filename).resolve())


def load_mesh(mesh_path: str) -> trimesh.Trimesh:
  loaded_mesh = trimesh.load(mesh_path)
  if isinstance(loaded_mesh, trimesh.Scene):
    if not loaded_mesh.geometry:
      raise ValueError(f"No geometry found in scene: {mesh_path}")
    return trimesh.util.concatenate(tuple(loaded_mesh.geometry.values()))
  return loaded_mesh


def rotate_and_plot_mesh(
  mesh_obj, 
  angle_x=0, 
  angle_y=0, 
  angle_z=0, 
  mesh_name="Mesh",
  alpha=0.95
):
  current_mesh = mesh_obj.copy()

  if angle_x:
    current_mesh.apply_transform(trimesh.transformations.rotation_matrix(np.radians(angle_x), [1, 0, 0]))
  if angle_y:
    current_mesh.apply_transform(trimesh.transformations.rotation_matrix(np.radians(angle_y), [0, 1, 0]))
  if angle_z:
    current_mesh.apply_transform(trimesh.transformations.rotation_matrix(np.radians(angle_z), [0, 0, 1]))

  fig = plt.figure(figsize=(6, 6))
  ax = fig.add_subplot(projection="3d")
  ax.add_collection3d(Poly3DCollection(current_mesh.triangles, alpha=alpha))

  min_coords = mesh_obj.vertices.min(axis=0)
  max_coords = mesh_obj.vertices.max(axis=0)
  max_range = float(np.max(max_coords - min_coords))
  if max_range == 0:
    max_range = 1.0
  midpoint = (min_coords + max_coords) / 2

  ax.set_xlim(midpoint[0] - max_range / 2, midpoint[0] + max_range / 2)
  ax.set_ylim(midpoint[1] - max_range / 2, midpoint[1] + max_range / 2)
  ax.set_zlim(midpoint[2] - max_range / 2, midpoint[2] + max_range / 2)
  ax.set_box_aspect([1, 1, 1])
  ax.set_title(f"{mesh_name} (X:{angle_x}°, Y:{angle_y}°, Z:{angle_z}°)")
  ax.axis("off")

  return fig


def create_interactive_mesh_plotter(
  mesh_obj, 
  mesh_name, 
  start_angles=(0, 0, 0), 
  step=5
):
  output_widget = Output()
  x_slider = IntSlider(
    min=0, 
    max=360, 
    step=step, 
    value=start_angles[0], 
    description="X (°):"
  )
  y_slider = IntSlider(
    min=0, 
    max=360, 
    step=step, 
    value=start_angles[1], 
    description="Y (°):"
  )
  z_slider = IntSlider(
    min=0, 
    max=360, 
    step=step, 
    value=start_angles[2], 
    description="Z (°):"
  )

  def update_plot(angle_x, angle_y, angle_z):
    with output_widget:
      clear_output(wait=True)
      fig = rotate_and_plot_mesh(
        mesh_obj,
        angle_x=angle_x,
        angle_y=angle_y,
        angle_z=angle_z,
        mesh_name=mesh_name,
      )
      display(fig)
      plt.close(fig)

  interactive(
    update_plot, 
    angle_x=x_slider, 
    angle_y=y_slider, 
    angle_z=z_slider
  )

  return VBox([
    x_slider,
    y_slider, 
    z_slider, 
    output_widget
  ])


def save_rotation_gif(
  mesh_obj, 
  mesh_name, 
  output_filename, 
  duration=10, 
  fps=20
):
  output_path = resolve_output_path(f"{output_filename}.gif")
  frame_count = duration * fps
  angles = np.linspace(0, 360, frame_count, endpoint=False)
  frames = []

  print(f"Generating GIF for {mesh_name}...")
  for angle in angles:
    fig = rotate_and_plot_mesh(
      mesh_obj, 
      angle_z=angle, 
      mesh_name=mesh_name
    )
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png")
    buffer.seek(0)
    frames.append(imageio.v2.imread(buffer))
    plt.close(fig)

  imageio.mimsave(output_path, frames, fps=fps)
  print(f"Saved GIF: {output_path}")
  display(Image(filename=output_path, width=420))
  return output_path

FileNotFoundError: sample_meshes not found in repository archive

## 1) MF Doom Mask

Load the mask mesh and run the shared GIF + interactive viewer pipeline.

In [5]:
# mask_filepath = resolve_output_path("low_poly_mf_doom_mask.glb")
mask_filepath = resolve_output_path("low_poly_mf_doom_mask.glb")
mf_doom_mask = load_mesh(mask_filepath)
mf_doom_mask_name = "MF Doom Mask"
mf_doom_fn = "mf_doom_mask_rotation_z_axis"

save_rotation_gif(
  mesh_obj=mf_doom_mask,
  mesh_name=mf_doom_mask_name,
  output_filename=mf_doom_fn,
  duration=10,
  fps=20
)

ValueError: string is not a file: `/content/sample_meshes/low_poly_mf_doom_mask.glb`

In [ ]:
display(
  create_interactive_mesh_plotter(
    mesh_obj=mf_doom_mask,
    mesh_name=mf_doom_mask_name
  )
)

## 2) Boolean Difference: Cube Minus Sphere

Create the boolean result mesh and reuse the same helper workflow.

In [ ]:
cube = trimesh.creation.box(extents=[10, 10, 10])
sphere = trimesh.creation.icosphere(radius=6.5, subdivisions=5)

cube_minus_sphere = trimesh.boolean.difference([cube, sphere])
if cube_minus_sphere is None:
  raise RuntimeError("Cube minus sphere boolean difference failed.")

cube_minus_sphere_name = "Cube - Sphere"
cube_minus_sphere_fn = "cube_minus_sphere.gif"

save_rotation_gif(
  mesh_obj=cube_minus_sphere,
  mesh_name=cube_minus_sphere_name,
  output_filename=cube_minus_sphere_fn,
  duration=10,
  fps=20
)

In [ ]:
display(
  create_interactive_mesh_plotter(
    mesh_obj=cube_minus_sphere,
    mesh_name=cube_minus_sphere_name
  )
)

## 3) Boolean Difference: Sphere Minus Cube

Reuse the same primitives and helpers for the inverse boolean subtraction.

The `cube` and `sphere` meshes from the previous cell are reused here.

In [ ]:
sphere_minus_cube = trimesh.boolean.difference([sphere, cube])
if sphere_minus_cube is None:
  raise RuntimeError("Sphere minus cube boolean difference failed.")

sphere_minus_cube_name = "Sphere - Cube"
sphere_minus_cube_fn = "sphere_minus_cube.gif"

save_rotation_gif(
  mesh_obj=sphere_minus_cube,
  mesh_name=sphere_minus_cube_name,
  output_filename=sphere_minus_cube_fn,
  duration=10,
  fps=20
)

In [ ]:
display(
  create_interactive_mesh_plotter(
    mesh_obj=sphere_minus_cube,
    mesh_name=sphere_minus_cube_name
  )
)

## 4) Convex-Hull Pyramid

Build a synthetic mesh and run it through the same shared GIF + interactive viewer utilities.

In [ ]:
vertices = np.array(
  [
    [-3, -3, 0],
    [ 3, -3, 0],
    [ 3,  3, 0],
    [-3,  3, 0],
    [ 0,  0, 3],
  ],
  dtype=float,
)
pyramid = trimesh.Trimesh(vertices=vertices, faces=ConvexHull(vertices).simplices)

save_rotation_gif(
  mesh_obj=pyramid,
  mesh_name="Pyramid",
  output_filename="pyramid_rotation_z_axis.gif",
  duration=10,
  fps=20
)

In [ ]:
display(
  create_interactive_mesh_plotter(
    mesh_obj=pyramid,
    mesh_name="Pyramid"
  )
)